In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import time


from utils_ML import score_dataset
from utils_preproces import ols_fit, y_distribution_plot, corr_heatmap_plot, make_mi_scores, preprocess_data

from sklearn.ensemble import RandomForestRegressor, VotingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor, plot_importance
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold


from sklearn.metrics import mean_absolute_error

In [2]:
# Set Matplotlib defaults
plt.style.use("seaborn-v0_8-white")
plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

In [3]:
# Fix random state
random_state = 42

In [4]:
harps = pd.read_csv('harps_sample.csv')
apogee = pd.read_csv('apogee_sample.csv')
galah = pd.read_csv('galah_sample.csv')
gr8 = pd.read_csv('gr8_not_in_harps_sample.csv')

print (f'harps contains {len(harps)} stars')
print (f'apogee contains {len(apogee)} stars')    
print (f'galah contains {len(galah)} stars')
print (f'gr8 contains {len(gr8)} stars')

harps contains 798 stars
apogee contains 108526 stars
galah contains 116050 stars
gr8 contains 1436 stars


In [5]:
apogee = apogee[apogee.parallax >2]
print (len(apogee))
galah = galah[galah.parallax > 3]
print (len(galah))


23511
19678


# Here we focus on getting Fe from no abundance

In [6]:
columns_for_Fe = ['radial_velocity', 'V_lsr',  'U_lsr',  'W_lsr', 'P_gal', # 'parallax',  #'pmra', 'pmdec', #'ra', 'dec',
                'fe_h', ]

apogee = apogee[columns_for_Fe]
harps = harps[columns_for_Fe]
galah = galah[columns_for_Fe]
gr8 = gr8[columns_for_Fe]

### Identifying and exluding null values

In [7]:
apogee.isnull().sum() # 196 null value for P_gal
apogee.dropna(inplace = True)

In [8]:
galah.isnull().sum() # 1 null value for P_gal
galah.dropna(inplace = True)

In [9]:
#gr8.isnull().sum() # No null values

In [10]:
#harps.isnull().sum() # No null values

### Spliting data into Trianing and Testing sets

In [11]:
# Split APOGEE
apogee_train_full, apogee_test = train_test_split(apogee, test_size=0.1, random_state=random_state)
# Split GALAH
galah_train_full, galah_test = train_test_split(galah, test_size=0.3, random_state=random_state)
# Combine APOGEE and GALAH train sets
combined_train = pd.concat([apogee_train_full, galah_train_full, gr8], ignore_index=True)


In [12]:

import h2o
from h2o.automl import H2OAutoML

h2o.init()

train_h2o = h2o.H2OFrame(combined_train)
test_h2o = h2o.H2OFrame(harps)

y_col = 'fe_h'  
x_cols = [col for col in train_h2o.columns if col != y_col]

aml = H2OAutoML(max_runtime_secs=1200, max_models=20, seed=random_state)
aml.train(x=x_cols, y=y_col, training_frame=train_h2o)

pred = aml.leader.predict(test_h2o).as_data_frame().values.flatten()
print("H2O AutoML MAE:", mean_absolute_error(harps['fe_h'], pred))



Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "21.0.7" 2025-04-15; OpenJDK Runtime Environment (build 21.0.7+6-Ubuntu-0ubuntu124.04); OpenJDK 64-Bit Server VM (build 21.0.7+6-Ubuntu-0ubuntu124.04, mixed mode, sharing)
  Starting server from /home/vadibekyan/anaconda3/lib/python3.12/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp34y00tsp
  JVM stdout: /tmp/tmp34y00tsp/h2o_vadibekyan_started_from_python.out
  JVM stderr: /tmp/tmp34y00tsp/h2o_vadibekyan_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Europe/Lisbon
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,2 months and 4 days
H2O_cluster_name:,H2O_from_python_vadibekyan_4d4phh
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,7.745 Gb
H2O_cluster_total_cores:,20
H2O_cluster_allowed_cores:,20
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
H2O AutoML MAE: 0.1727371329524352


/home/vadibekyan/anaconda3/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [14]:
# Show the leaderboard
lb = aml.leaderboard
leaderboard_df = lb.as_data_frame()
leaderboard_df


/home/vadibekyan/anaconda3/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,model_id,rmse,mse,mae,rmsle,mean_residual_deviance
0,StackedEnsemble_AllModels_1_AutoML_1_20250601_...,0.180142,0.032451,0.139136,NaN,0.032451
1,StackedEnsemble_BestOfFamily_1_AutoML_1_202506...,0.180228,0.032482,0.139158,NaN,0.032482
2,GBM_1_AutoML_1_20250601_111134,0.180668,0.032641,0.139483,NaN,0.032641
3,GBM_2_AutoML_1_20250601_111134,0.180902,0.032726,0.139692,NaN,0.032726
4,GBM_5_AutoML_1_20250601_111134,0.180925,0.032734,0.139754,NaN,0.032734
5,DeepLearning_grid_3_AutoML_1_20250601_111134_m...,0.180926,0.032734,0.139838,NaN,0.032734
6,GBM_grid_1_AutoML_1_20250601_111134_model_1,0.180958,0.032746,0.139873,NaN,0.032746
7,GBM_3_AutoML_1_20250601_111134,0.181042,0.032776,0.139771,NaN,0.032776
8,DeepLearning_grid_2_AutoML_1_20250601_111134_m...,0.181170,0.032822,0.139888,NaN,0.032822
9,GBM_4_AutoML_1_20250601_111134,0.181180,0.032826,0.139790,NaN,0.032826


In [ ]:
#best_model = h2o.get_model(leaderboard_df.loc[0, 'model_id'])
#best_model.params
#aml.leader #gives the best model
# Plot variable importance
#best_model.varimp_plot()

key,value
Stacking strategy,cross_validation
Number of base models (used / total),13/38
# GBM base models (used / total),10/31
# DeepLearning base models (used / total),2/4
# DRF base models (used / total),1/2
# GLM base models (used / total),0/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [15]:
from autogluon.tabular import TabularPredictor

# Combine features and target into one DataFrame
train_df = combined_train
test_df = harps

# Create predictor
predictor = TabularPredictor(label='fe_h', eval_metric='mae').fit(train_df, time_limit=4*3600)
# Predict on test set
y_pred = predictor.predict(test_df)

# Evaluate
from sklearn.metrics import mean_absolute_error
print("AutoGluon MAE:", mean_absolute_error(harps['fe_h'], y_pred))


No path specified. Models will be saved in: "AutogluonModels/ag-20250601_103739"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.7
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #26~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Apr 17 19:20:47 UTC 2
CPU Count:          20
Memory Avail:       21.22 GB / 31.05 GB (68.3%)
Disk Space Avail:   546.08 GB / 936.79 GB (58.3%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use 

AutoGluon MAE: 0.17492070651376265


In [17]:
predictor.leaderboard(silent=False) # shows performance of the models

                  model  score_val          eval_metric  pred_time_val     fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2  -0.136601  mean_absolute_error       0.599153  1629.553190                0.000648           0.086880            2       True         12
1        NeuralNetTorch  -0.137298  mean_absolute_error       0.095239  1617.060272                0.095239        1617.060272            1       True         10
2               XGBoost  -0.137463  mean_absolute_error       0.016342     3.150049                0.016342           3.150049            1       True          9
3              LightGBM  -0.137560  mean_absolute_error       0.009758     3.634269                0.009758           3.634269            1       True          4
4              CatBoost  -0.137725  mean_absolute_error       0.001537     1.278360                0.001537           1.278360            1       True          6
5            LightGBMXT  -0.

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.136601,mean_absolute_error,0.599153,1629.553190,0.000648,0.086880,2,True,12
1,NeuralNetTorch,-0.137298,mean_absolute_error,0.095239,1617.060272,0.095239,1617.060272,1,True,10
2,XGBoost,-0.137463,mean_absolute_error,0.016342,3.150049,0.016342,3.150049,1,True,9
3,LightGBM,-0.137560,mean_absolute_error,0.009758,3.634269,0.009758,3.634269,1,True,4
4,CatBoost,-0.137725,mean_absolute_error,0.001537,1.278360,0.001537,1.278360,1,True,6
5,LightGBMXT,-0.137839,mean_absolute_error,0.041083,11.484606,0.041083,11.484606,1,True,3
6,LightGBMLarge,-0.137908,mean_absolute_error,0.018762,14.506944,0.018762,14.506944,1,True,11
7,NeuralNetFastAI,-0.138189,mean_absolute_error,0.154014,187.865149,0.154014,187.865149,1,True,8
8,RandomForestMSE,-0.139908,mean_absolute_error,0.326601,7.520315,0.326601,7.520315,1,True,5
9,ExtraTreesMSE,-0.139936,mean_absolute_error,0.100296,1.625245,0.100296,1.625245,1,True,7


In [19]:
leaderboard_df = predictor.leaderboard(silent=True)
best_model = leaderboard_df.iloc[0]['model']
print("Best model:", best_model)
y_pred = predictor.predict(test_df, model=best_model)
print("MAE:", mean_absolute_error(test_df['fe_h'], y_pred))
performance = predictor.evaluate_predictions(
    y_true=test_df['fe_h'],
    y_pred=predictor.predict(test_df, model='XGBoost'),
    auxiliary_metrics=True
)
print(performance)

Best model: WeightedEnsemble_L2
MAE: 0.17492070651376265
{'mean_absolute_error': -0.17347792863207126, 'root_mean_squared_error': -0.22206456612562897, 'mean_squared_error': -0.04931267152856384, 'r2': 0.26705160346656254, 'pearsonr': 0.5659787601558297, 'median_absolute_error': -0.14245696628466248}
